# 08 - Reserved Capacity Sensitivity

This notebook asks whether `Q`, the number of Class-1-reserved slots per day, is the main control parameter for each corrected reservation policy.

Policies in focus:

- **Strict C1 reservation:** `Q` slots per day are reserved for Class 1. Class 1 searches appointment days chronologically, trying reserved capacity before general capacity within the same day. Class 2 can only use general capacity. Unused reserved slots stay empty.
- **Class-1-first backfill reservation:** each simulation day, Class 1 arrivals are processed before Class 2 arrivals. Class 1 searches chronologically, trying reserved before general within the same appointment day. After the Class 1 batch, Class 2 searches general capacity first, then leftover reserved capacity.
- **Pooled FCFS:** no reserved slots. It appears only as a light horizontal reference baseline.

## Setup

Use the same simple repo-root detection and imports as notebook 07.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "simulation" / "engine.py").exists() and (
            candidate / "analysis" / "metrics.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import PatientClassParams, SimulationConfig, ThresholdRule

plt.style.use("default")

## Base Scenario

The demand level is fixed at `lambda_1 = 25` and `lambda_2 = 25`. The sweep changes only the number of reserved Class 1 slots `Q`.

In [ ]:
BASE_SCENARIO = {
    "slots_per_day": 32,
    "reserved_slots_per_day": 10,
    "reserved_class_id": 1,
    "horizon_days": 14,
    "burn_in_days": 30,
    "measure_days": 365,
    "cooldown_days": 14,
    "seeds": list(range(5101, 5131)),
    "classes": {
        1: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
        2: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
    },
}

Q_VALUES = [0, 2, 4, 6, 8, 10, 12, 16, 20, 24, 28, 32]
RESERVATION_POLICIES = ["Strict C1 reservation", "Class-1-first backfill reservation"]
POLICY_COLORS = {
    "Strict C1 reservation": "tab:blue",
    "Class-1-first backfill reservation": "tab:orange",
    "Pooled FCFS": "0.45",
}
LOST_COMPONENTS = [
    "balked_rate",
    "canceled_rate",
    "no_show_rate",
    "no_offer_rate",
    "unresolved_booked_rate",
]

scenario_table = pd.DataFrame(
    {
        "value": pd.Series(
            {
                key: value
                for key, value in BASE_SCENARIO.items()
                if key not in {"classes", "seeds"}
            },
            dtype="object",
        )
    }
)
scenario_table.loc["num_seeds", "value"] = len(BASE_SCENARIO["seeds"])
scenario_table.loc["seed_range", "value"] = f"{BASE_SCENARIO['seeds'][0]}-{BASE_SCENARIO['seeds'][-1]}"
scenario_table.loc["q_values", "value"] = Q_VALUES

class_table = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "lambda_per_day": params["lambda_per_day"],
            "cancel_prob": params["cancel_prob"],
            "balk_threshold": params["balk_prob"]["threshold"],
            "balk_low": params["balk_prob"]["low"],
            "balk_high": params["balk_prob"]["high"],
            "no_show_threshold": params["no_show_prob"]["threshold"],
            "no_show_low": params["no_show_prob"]["low"],
            "no_show_high": params["no_show_prob"]["high"],
        }
        for class_id, params in BASE_SCENARIO["classes"].items()
    ]
)

display(scenario_table)
display(class_table)

## Small Helpers

The helpers below are intentionally narrow: build one simulation config, update `Q`, run the Q sweep, add outcome rates, and check accounting.

In [ ]:
def build_config(scenario: dict, policy: str, seed: int | None) -> SimulationConfig:
    classes = {}
    for class_id, params in scenario["classes"].items():
        classes[class_id] = PatientClassParams(
            class_id=class_id,
            lambda_per_day=float(params["lambda_per_day"]),
            balk_prob=ThresholdRule(**params["balk_prob"]),
            cancel_prob=float(params["cancel_prob"]),
            no_show_prob=ThresholdRule(**params["no_show_prob"]),
            value=float(params.get("value", 1.0)),
        )

    reserved = policy != "Pooled FCFS"
    backfill_reserved = policy == "Class-1-first backfill reservation"
    reserved_slots = int(scenario["reserved_slots_per_day"]) if reserved else 0

    return SimulationConfig(
        slots_per_day=int(scenario["slots_per_day"]),
        horizon_days=int(scenario["horizon_days"]),
        burn_in_days=int(scenario["burn_in_days"]),
        measure_days=int(scenario["measure_days"]),
        cooldown_days=int(scenario["cooldown_days"]),
        classes=classes,
        seed=seed,
        reserved_class_id=scenario["reserved_class_id"] if reserved_slots > 0 else None,
        reserved_slots_per_day=reserved_slots,
        release_reserved_slots=backfill_reserved and reserved_slots > 0,
    )


def make_scenario(q: int, base_scenario: dict = BASE_SCENARIO) -> dict:
    scenario = deepcopy(base_scenario)
    scenario["reserved_slots_per_day"] = int(q)
    return scenario


def add_rates(aggregate_df: pd.DataFrame, class_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    aggregate_df = aggregate_df.copy()
    class_df = class_df.copy()

    arrivals = aggregate_df["total_arrivals"]
    aggregate_df["served_rate"] = aggregate_df["total_served"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["balked_rate"] = aggregate_df["total_balked"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["no_offer_rate"] = aggregate_df["total_no_offer"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["canceled_rate"] = aggregate_df["total_canceled"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["no_show_rate"] = aggregate_df["total_no_show"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["unresolved_booked_rate"] = aggregate_df["total_unresolved_booked"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["lost_components_sum"] = aggregate_df[LOST_COMPONENTS].sum(axis=1)
    aggregate_df["lost_rate"] = 1.0 - aggregate_df["served_rate"]

    class_df["unresolved_booked"] = class_df["booked"] - class_df["canceled"] - class_df["no_show"] - class_df["served"]
    class_arrivals = class_df["arrivals"]
    class_df["served_rate"] = class_df["percent_serviced"]
    class_df["balked_rate"] = class_df["balked"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["no_offer_rate"] = class_df["no_offer"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["canceled_rate"] = class_df["canceled"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["no_show_rate"] = class_df["no_show"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["unresolved_booked_rate"] = class_df["unresolved_booked"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["lost_components_sum"] = class_df[LOST_COMPONENTS].sum(axis=1)
    class_df["lost_rate"] = 1.0 - class_df["served_rate"]

    return aggregate_df, class_df


def validate_accounting(aggregate_df: pd.DataFrame, class_df: pd.DataFrame, tol: float = 1e-9) -> None:
    aggregate_partition = (
        aggregate_df["total_served"]
        + aggregate_df["total_balked"]
        + aggregate_df["total_no_offer"]
        + aggregate_df["total_canceled"]
        + aggregate_df["total_no_show"]
        + aggregate_df["total_unresolved_booked"]
    )
    if (aggregate_partition - aggregate_df["total_arrivals"]).abs().max() > tol:
        raise AssertionError("Aggregate outcomes do not partition arrivals.")
    if (aggregate_df["total_unresolved_booked"] < -tol).any():
        raise AssertionError("Aggregate unresolved_booked is negative.")
    if (aggregate_df["lost_rate"] - (1.0 - aggregate_df["served_rate"])).abs().max() > tol:
        raise AssertionError("Aggregate lost_rate is not 1 - served_rate.")
    if (aggregate_df["lost_components_sum"] - aggregate_df["lost_rate"]).abs().max() > tol:
        raise AssertionError("Aggregate lost components do not sum to lost_rate.")

    class_partition = (
        class_df["served"]
        + class_df["balked"]
        + class_df["no_offer"]
        + class_df["canceled"]
        + class_df["no_show"]
        + class_df["unresolved_booked"]
    )
    if (class_partition - class_df["arrivals"]).abs().max() > tol:
        raise AssertionError("Class outcomes do not partition arrivals.")
    if (class_df["unresolved_booked"] < -tol).any():
        raise AssertionError("Class unresolved_booked is negative.")
    if (class_df["lost_rate"] - (1.0 - class_df["served_rate"])).abs().max() > tol:
        raise AssertionError("Class lost_rate is not 1 - served_rate.")
    if (class_df["lost_components_sum"] - class_df["lost_rate"]).abs().max() > tol:
        raise AssertionError("Class lost components do not sum to lost_rate.")


def run_q_sweep(q_values: list[int]) -> tuple[pd.DataFrame, pd.DataFrame]:
    aggregate_rows = []
    class_rows = []

    for q in tqdm(q_values, desc="Q values"):
        scenario = make_scenario(q)
        for policy in RESERVATION_POLICIES:
            for seed in scenario["seeds"]:
                config = build_config(scenario, policy, seed=int(seed))
                result = ClinicAppointmentSimulation(config).run()
                fixed_values = {"policy": policy, "seed": int(seed), "Q": int(q)}
                aggregate_rows.append(aggregate_result_row(result, fixed_values))
                class_rows.extend(class_result_rows(result, fixed_values))

    aggregate_df = pd.DataFrame(aggregate_rows)
    class_df = pd.DataFrame(class_rows)
    aggregate_df, class_df = add_rates(aggregate_df, class_df)
    validate_accounting(aggregate_df, class_df)
    return aggregate_df, class_df

## Q Sweep

Run strict reservation and Class-1-first backfill reservation for each `Q`. Pooled FCFS is run once at `Q = 0` and used only as a reference.

In [ ]:
q_aggregate_df, q_class_df = run_q_sweep(Q_VALUES)

fcfs_aggregate_rows = []
fcfs_class_rows = []
fcfs_scenario = make_scenario(0)
for seed in tqdm(fcfs_scenario["seeds"], desc="Pooled FCFS reference"):
    config = build_config(fcfs_scenario, "Pooled FCFS", seed=int(seed))
    result = ClinicAppointmentSimulation(config).run()
    fixed_values = {"policy": "Pooled FCFS", "seed": int(seed), "Q": 0}
    fcfs_aggregate_rows.append(aggregate_result_row(result, fixed_values))
    fcfs_class_rows.extend(class_result_rows(result, fixed_values))

fcfs_aggregate_df = pd.DataFrame(fcfs_aggregate_rows)
fcfs_class_df = pd.DataFrame(fcfs_class_rows)
fcfs_aggregate_df, fcfs_class_df = add_rates(fcfs_aggregate_df, fcfs_class_df)
validate_accounting(fcfs_aggregate_df, fcfs_class_df)

print(f"Reservation-policy runs: {len(q_aggregate_df):,}")
print(f"FCFS reference runs: {len(fcfs_aggregate_df):,}")

## Summary Tables

In [ ]:
aggregate_columns = [
    "served_rate",
    "average_utilization",
    "mean_offered_booking_delay",
    "lost_rate",
    "balked_rate",
    "no_offer_rate",
    "canceled_rate",
    "no_show_rate",
    "unresolved_booked_rate",
]
class_columns = [
    "served_rate",
    "mean_offered_booking_delay",
    "lost_rate",
    "balked_rate",
    "no_offer_rate",
    "canceled_rate",
    "no_show_rate",
    "unresolved_booked_rate",
]

aggregate_means = (
    q_aggregate_df.groupby(["Q", "policy"])[aggregate_columns]
    .mean()
    .round(4)
)
class_means = (
    q_class_df.groupby(["Q", "policy", "class_id"])[class_columns]
    .mean()
    .round(4)
)
fcfs_aggregate_reference = (
    fcfs_aggregate_df.groupby("policy")[aggregate_columns]
    .mean()
    .round(4)
)
fcfs_class_reference = (
    fcfs_class_df.groupby(["policy", "class_id"])[class_columns]
    .mean()
    .round(4)
)

display(aggregate_means)
display(class_means)
print("Pooled FCFS reference")
display(fcfs_aggregate_reference)
display(fcfs_class_reference)

## Main Plots

In [ ]:
# aggregate_plot = (
#     q_aggregate_df.groupby(["Q", "policy"])[aggregate_columns]
#     .mean()
#     .reset_index()
# )
# class_plot = (
#     q_class_df.groupby(["Q", "policy", "class_id"])[class_columns]
#     .mean()
#     .reset_index()
# )
# fcfs_aggregate_plot = fcfs_aggregate_df[aggregate_columns].mean()
# fcfs_class_plot = fcfs_class_df.groupby("class_id")[class_columns].mean()

# fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# ax = axes[0, 0]
# for policy in RESERVATION_POLICIES:
#     for class_id, linestyle in [(1, "-"), (2, "--")]:
#         subset = class_plot[(class_plot["policy"] == policy) & (class_plot["class_id"] == class_id)]
#         ax.plot(
#             subset["Q"],
#             subset["served_rate"],
#             marker="o",
#             linestyle=linestyle,
#             color=POLICY_COLORS[policy],
#             label=f"{policy}, C{class_id}",
#         )
# for class_id, linestyle in [(1, ":"), (2, "-.")]:
#     ax.axhline(
#         fcfs_class_plot.loc[class_id, "served_rate"],
#         color="0.45",
#         linestyle=linestyle,
#         linewidth=1.4,
#         alpha=0.85,
#         label=f"FCFS C{class_id} reference",
#     )
# ax.set_title("Served rate by class vs Q")
# ax.set_xlabel("Reserved slots per day, Q")
# ax.set_ylabel("Served rate")
# ax.grid(axis="y", alpha=0.25)
# ax.legend(fontsize=8)

# ax = axes[0, 1]
# for policy in RESERVATION_POLICIES:
#     for class_id, linestyle in [(1, "-"), (2, "--")]:
#         subset = class_plot[(class_plot["policy"] == policy) & (class_plot["class_id"] == class_id)]
#         ax.plot(
#             subset["Q"],
#             subset["mean_offered_booking_delay"],
#             marker="o",
#             linestyle=linestyle,
#             color=POLICY_COLORS[policy],
#             label=f"{policy}, C{class_id}",
#         )
# for class_id, linestyle in [(1, ":"), (2, "-.")]:
#     ax.axhline(
#         fcfs_class_plot.loc[class_id, "mean_offered_booking_delay"],
#         color="0.45",
#         linestyle=linestyle,
#         linewidth=1.4,
#         alpha=0.85,
#         label=f"FCFS C{class_id} reference",
#     )
# ax.set_title("Mean offered delay by class vs Q")
# ax.set_xlabel("Reserved slots per day, Q")
# ax.set_ylabel("Mean offered booking delay")
# ax.grid(axis="y", alpha=0.25)
# ax.legend(fontsize=8)

# ax = axes[1, 0]
# for policy in RESERVATION_POLICIES:
#     subset = aggregate_plot[aggregate_plot["policy"] == policy]
#     ax.plot(
#         subset["Q"],
#         subset["average_utilization"],
#         marker="o",
#         color=POLICY_COLORS[policy],
#         label=policy,
#     )
# ax.axhline(
#     fcfs_aggregate_plot["average_utilization"],
#     color="0.45",
#     linestyle="--",
#     linewidth=1.4,
#     alpha=0.85,
#     label="FCFS reference",
# )
# ax.set_title("Average utilization vs Q")
# ax.set_xlabel("Reserved slots per day, Q")
# ax.set_ylabel("Average utilization")
# ax.grid(axis="y", alpha=0.25)
# ax.legend(fontsize=8)

# class_served_wide = class_plot.pivot(index=["Q", "policy"], columns="class_id", values="served_rate").reset_index()
# class_served_wide["priority_gap"] = class_served_wide[1] - class_served_wide[2]
# tradeoff_plot = class_served_wide.merge(
#     aggregate_plot[["Q", "policy", "served_rate"]],
#     on=["Q", "policy"],
#     how="left",
# )

# ax = axes[1, 1]
# for policy in RESERVATION_POLICIES:
#     subset = tradeoff_plot[tradeoff_plot["policy"] == policy].sort_values("Q")
#     ax.plot(
#         subset["priority_gap"],
#         subset["served_rate"],
#         marker="o",
#         color=POLICY_COLORS[policy],
#         label=policy,
#     )
#     for _, row in subset.iterrows():
#         ax.text(row["priority_gap"], row["served_rate"], f" {int(row['Q'])}", fontsize=8)
# ax.set_title("Overall served rate vs priority gap")
# ax.set_xlabel("Class 1 served rate - Class 2 served rate")
# ax.set_ylabel("Overall served rate")
# ax.grid(axis="y", alpha=0.25)
# ax.legend(fontsize=8)

# fig.tight_layout()

In [ ]:
reserved_slot_label = "Reserved Class 1 slots per day"

aggregate_plot = (
    q_aggregate_df.groupby(["Q", "policy"])[aggregate_columns]
    .mean()
    .reset_index()
)

class_plot = (
    q_class_df.groupby(["Q", "policy", "class_id"])[class_columns]
    .mean()
    .reset_index()
)

fcfs_aggregate_plot = fcfs_aggregate_df[aggregate_columns].mean()
fcfs_class_plot = fcfs_class_df.groupby("class_id")[class_columns].mean()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for policy in RESERVATION_POLICIES:
    for class_id, linestyle in [(1, "-"), (2, "--")]:
        subset = class_plot[
            (class_plot["policy"] == policy)
            & (class_plot["class_id"] == class_id)
        ].sort_values("Q")

        ax.plot(
            subset["Q"],
            subset["served_rate"],
            marker="o",
            linestyle=linestyle,
            color=POLICY_COLORS[policy],
            label=f"{policy}, Class {class_id}",
        )

for class_id, linestyle in [(1, ":"), (2, "-.")]:
    ax.axhline(
        fcfs_class_plot.loc[class_id, "served_rate"],
        color="0.45",
        linestyle=linestyle,
        linewidth=1.4,
        alpha=0.85,
        label=f"FCFS Class {class_id} reference",
    )

ax.set_title("Served Rate By Class")
ax.set_xlabel(reserved_slot_label)
ax.set_ylabel("Served rate")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for policy in RESERVATION_POLICIES:
    for class_id, linestyle in [(1, "-"), (2, "--")]:
        subset = class_plot[
            (class_plot["policy"] == policy)
            & (class_plot["class_id"] == class_id)
        ].sort_values("Q")

        ax.plot(
            subset["Q"],
            subset["mean_offered_booking_delay"],
            marker="o",
            linestyle=linestyle,
            color=POLICY_COLORS[policy],
            label=f"{policy}, Class {class_id}",
        )

for class_id, linestyle in [(1, ":"), (2, "-.")]:
    ax.axhline(
        fcfs_class_plot.loc[class_id, "mean_offered_booking_delay"],
        color="0.45",
        linestyle=linestyle,
        linewidth=1.4,
        alpha=0.85,
        label=f"FCFS Class {class_id} reference",
    )

ax.set_title("Mean Offered Delay By Class")
ax.set_xlabel(reserved_slot_label)
ax.set_ylabel("Mean offered booking delay")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for policy in RESERVATION_POLICIES:
    subset = aggregate_plot[
        aggregate_plot["policy"] == policy
    ].sort_values("Q")

    ax.plot(
        subset["Q"],
        subset["average_utilization"],
        marker="o",
        color=POLICY_COLORS[policy],
        label=policy,
    )

ax.axhline(
    fcfs_aggregate_plot["average_utilization"],
    color="0.45",
    linestyle="--",
    linewidth=1.4,
    alpha=0.85,
    label="FCFS reference",
)

ax.set_title("Average Utilization")
ax.set_xlabel(reserved_slot_label)
ax.set_ylabel("Average utilization")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

fig.tight_layout()

In [ ]:
class_served_wide = (
    class_plot
    .pivot(index=["Q", "policy"], columns="class_id", values="served_rate")
    .reset_index()
)

class_served_wide["priority_gap"] = (
    class_served_wide[1] - class_served_wide[2]
)

tradeoff_plot = class_served_wide.merge(
    aggregate_plot[["Q", "policy", "served_rate"]],
    on=["Q", "policy"],
    how="left",
)

fig, ax = plt.subplots(figsize=(8, 5))

for policy in RESERVATION_POLICIES:
    subset = tradeoff_plot[
        tradeoff_plot["policy"] == policy
    ].sort_values("Q")

    ax.plot(
        subset["priority_gap"],
        subset["served_rate"],
        marker="o",
        color=POLICY_COLORS[policy],
        label=policy,
    )

    for _, row in subset.iterrows():
        ax.text(
            row["priority_gap"],
            row["served_rate"],
            f" {int(row['Q'])}",
            fontsize=8,
        )

ax.set_title("Overall Served Rate vs Class Priority Gap")
ax.set_xlabel("Class 1 served rate minus Class 2 served rate")
ax.set_ylabel("Overall served rate")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

fig.tight_layout()

## Short Interpretation Placeholders

- Low Q:
- Intermediate Q:
- High Q:
- Difference between strict and Class-1-first backfill:

## Accounting Checks

The sweep and FCFS reference both call `validate_accounting` immediately after running. This final cell repeats the checks explicitly.

In [ ]:
for q in Q_VALUES:
    validate_accounting(
        q_aggregate_df[q_aggregate_df["Q"] == q],
        q_class_df[q_class_df["Q"] == q],
    )
validate_accounting(fcfs_aggregate_df, fcfs_class_df)

print("Accounting checks passed for every Q and the FCFS reference.")

## Final Note

This notebook feeds the Q-sensitivity figure in the short findings document. It is not a full behavioral sensitivity analysis.